<span id="isaac-lab과-robot-simulation"></span>

# Isaac Lab and robot simulation

<span id="학습-목표"></span>

## Learning objectives

Go beyond running a new simulator example: Identify the observations, actions, and physics that your research task requires you to change.

<span id="실행-조건"></span>

## Execution conditions

- CPU path: Gymnasium, NumPy, and Matplotlib.
- Official SO-101 tutorial: Python 3.12, uv, Git LFS, Linux/Windows + NVIDIA GPU.
- GPU environment: Check before class. Prefer personal equipment, supplemented by shared resources.
- Jetson extension: edge inference and latency measurement.

<span id="적용-사례"></span>

## Applications

<span id="이동로봇의-경로와-안전"></span>

### Mobile robot paths and safety

An application design case choosing either grid paths or continuous velocity control.

**Observation and state:** Position, target-relative position, and nearby obstacle observations.

**Action:** choose a grid direction or bounded linear and angular velocities.

**Reward and constraints:** Separate arrival, collision, and movement costs. Distinguish collisions from timeouts.

**Comparison baseline:** shortest path and rule-based avoidance.

**Metrics to check:** Success rate, collision rate, travel distance, and time. Evaluate on unseen maps.

<span id="구동계의-추종-성능과-입력-비용"></span>

### Drive-system tracking performance and input cost

A synthetic 1st-order motor model that tracks a constant reference angular velocity in each experiment. Separate training at target 1 from evaluation at 0.5, 1, and 1.5.

**Observation and state:** angular velocity $\omega/5$, target $r/2$.

**Action:** Continuous $a\in[-1,1]$, actual input $u=2a$.

**Reward and constraints:** $-((r-\omega)^2+0.05u^2)$. Termination at $|\omega|>5$; time limit at 100 steps.

**Comparison baseline:** P control with the same input limits.

**Metrics to check:** Tracking RMSE, mean u², and number of safety terminations. Evaluate separately for each target.

<span id="gymnasium에서-로봇-simulator로"></span>

## From Gymnasium to a robot simulator

<span id="시작-조건"></span>

### Starting conditions

Start from CartPole-v1 observations, actions, resets, and time limits.

<span id="진행-방법"></span>

### Procedure

1.  Run the provided rule policy and record observations and actions for one episode.
2.  In the official Isaac Lab Quickstart, identify and map the roles of the task, physics backend, renderer, and RL library.

<span id="남길-결과"></span>

### Results to retain

- Environment interface mapping table and one rollout.

In [ ]:
"""Small public-environment benchmark; evaluation traces are ordinary numeric arrays."""
import json
from importlib.metadata import version
from pathlib import Path
import numpy as np
import gymnasium as gym

def collect(env_id='CartPole-v1', policy=None, seeds=range(100,103)):
    env=gym.make(env_id); transitions=[]; summary=[]
    for seed in seeds:
        obs,info=env.reset(seed=seed);env.action_space.seed(seed);total=0.;length=0
        while True:
            action=env.action_space.sample() if policy is None else policy(obs)
            nxt,reward,terminated,truncated,info=env.step(action)
            transitions.append((obs.copy(),np.asarray(action).copy(),reward,nxt.copy(),terminated,truncated,seed))
            obs=nxt;total+=reward;length+=1
            if terminated or truncated:break
        summary.append([seed,total,length,int(terminated),int(truncated)])
    env.close()
    columns=list(zip(*transitions))
    data={k:np.asarray(v) for k,v in zip(['observations','actions','rewards','next_observations','terminated','truncated','episode_seed'],columns)}
    return np.asarray(summary),data

def cartpole_rule(obs):return int(obs[2]+.25*obs[3]>0)
def zero_torque(obs):return np.zeros(1,dtype=np.float32)

def save_trace(path, env_id, data, policy_name, seeds):
    # Contains numeric arrays and JSON text only; never requires pickle.
    metadata=dict(env_id=env_id,policy=policy_name,seeds=list(seeds),gymnasium=version('gymnasium'))
    np.savez_compressed(path,**data,metadata_json=np.array(json.dumps(metadata)))
    with np.load(path,allow_pickle=False) as saved:
        assert len(saved['observations'])==len(saved['rewards'])
        print(json.loads(str(saved['metadata_json'])))

def public_model(algorithm='DQN',env_id='CartPole-v1',seed=7,steps=2048,**changes):
    from stable_baselines3 import DQN,PPO,SAC
    options=dict(policy_kwargs=dict(net_arch=[32,32]),seed=seed,verbose=0,device='cpu')
    if algorithm=='DQN':options.update(buffer_size=5000,learning_starts=100,target_update_interval=100,batch_size=32)
    if algorithm=='PPO':options.update(n_steps=128,batch_size=64,n_epochs=3)
    if algorithm=='SAC':options.update(buffer_size=5000,learning_starts=100,batch_size=64)
    options.update(changes)
    env=gym.make(env_id)
    model={'DQN':DQN,'PPO':PPO,'SAC':SAC}[algorithm]('MlpPolicy',env,**options)
    model.learn(total_timesteps=steps)
    env.close()
    return model

print('Gymnasium',version('gymnasium'))
env=gym.make('CartPole-v1')
print('Observation:',env.observation_space,'Action:',env.action_space,'Limit:',env.spec.max_episode_steps)
env.close()
# For visual inspection, create a separate env with render_mode='rgb_array'.
# Rendering requires gymnasium[classic-control]; do not render every training step.

rows,trace=collect(policy=cartpole_rule,seeds=[100]);print(rows)


<span id="결과-기록자가-점검"></span>

## Record results and self-check

**Role:**

**Gymnasium example:**

**Isaac Lab configuration:**

**What I will change:**

<span id="확인할-점"></span>

### Points to check

- Did you distinguish the physics simulation from the learning algorithm as separate components?
- Did you distinguish the policy control period from the physics simulation period?

<span id="환경-온보딩-실행-또는-transition-분석"></span>

## Environment onboarding: execution or transition analysis

<span id="시작-조건-1"></span>

### Starting conditions

GPU environment: On a prepared Linux/Windows NVIDIA system, follow the official Quickstart in this order: task list → zero/random agent → training → play. CPU environment: During the same session, complete CartPole transition and policy comparisons and task configuration mapping.

<span id="진행-방법-1"></span>

### Procedure

1.  If a GPU environment is ready, run the Cartpole task following the current official documentation, and record the task ID, backend, seed, and execution command.
2.  If no GPU environment is available, compare Random and Rule trajectories with the provided CPU code and identify termination causes.
3.  In your selected path, find the setting to modify to change one observation, action, or termination rule.

<span id="남길-결과-1"></span>

### Results to retain

- GPU: Run screenshot or rollout and execution conditions. CPU: State trajectory plot and configuration mapping table.

In [ ]:
import matplotlib.pyplot as plt
for name,policy in [('random',None),('rule',cartpole_rule)]:
    rows,trace=collect(policy=policy,seeds=[100]);plt.plot(trace['observations'][:,2],label=name)
plt.xlabel('Control step');plt.ylabel('Pole angle [rad]');plt.legend()


<span id="결과-기록자가-점검-1"></span>

## Record results and self-check

**Path:**

**Task and environment:**

**Execution conditions:**

**Observed results:**

**Settings to modify:**

<span id="확인할-점-1"></span>

### Points to check

- Did you avoid presenting unexecuted Isaac results as actual run results?
- Did you trace observations, actions, and terminations beyond merely resolving installation issues?

<span id="연구-task의-최소-구성"></span>

## Minimum components of a research task

<span id="시작-조건-2"></span>

### Starting conditions

Choose manipulation, mobile robotics, or process control.

<span id="진행-방법-2"></span>

### Procedure

1.  Tabulate the scene, state observations, action scale, reward, reset, and termination.
2.  Link to the files and components containing these settings in the official Isaac Lab robot tutorial.
3.  Decide what to hold fixed and which single item to change in the first experiment.

<span id="남길-결과-2"></span>

### Results to retain

- Minimum task design table and initial validation plan.

<span id="결과-기록자가-점검-2"></span>

## Record results and self-check

**Components:**

**My task definition:**

**Reference configuration location:**

**Initial validation:**

<span id="확인할-점-2"></span>

### Points to check

- Did you distinguish successful visualization from a correctly defined learning problem?
- Did you identify the core issue to validate rather than trying to reproduce a complete robot system from the start?

<span id="공식-예제데이터"></span>

## Official examples and data

- [Isaac Lab · Quickstart](https://isaac-sim.github.io/IsaacLab/main/source/isaaclab/quickstart.html)
- [Isaac Lab · SO-101 task tutorial](https://isaac-sim.github.io/IsaacLab/develop/source/isaaclab/tutorial.html)